# IE 643 - Experiment E2

**Question:** If we fix the seed and change one word of the prompt, does the rest of the clip stay put?

**Config:** AnimateDiff v1.5 + `mm_sd_v15_v2`, SD1.5 backbone. 16 frames, 512x512,
25 DDIM steps, CFG 7.5 - identical to E1. All 7 clips share **one fixed noise tensor**
(roadmap S2.3 requires the tensor, not just the seed).

**Design:** 1 base clip + 3 noun swaps (cat / horse / bird, ordered by shape-distance)
+ 3 verb swaps (walking / jumping / standing).

**Pass condition:** mean `S` is **positive** for noun swaps and **negative** for verb swaps.

---

### Kaggle setup

1. Right sidebar -> **Accelerator** -> `GPU T4 x2`
2. Right sidebar -> **Internet** -> **On**

Runtime ~11 min for the 7 clips (+ ~15 s to build the pipeline, + ~1 min for CLIP on first use).
Cell 1 is self-contained: it runs in a fresh kernel, and reuses `pipe` if E1 left one in memory.


In [ ]:
# ============ E2 : seed / noise control  (SELF-CONTAINED) ============
# Q: if we fix the seed and change ONE word, does the rest of the clip stay put?
# Safe to run in a fresh kernel: defines its own builder and reuses `pipe` if present.

import sys, subprocess, importlib
try:
    import diffusers, transformers
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "diffusers", "transformers", "accelerate"], check=True)
    import diffusers, transformers

import torch, gc, time, json, numpy as np
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler

FRAMES, H, W, STEPS, CFG = 16, 512, 512, 25, 7.5
SEED, SCENE = 42, "on a beach"
NEGATIVE = "bad quality, worse quality, blurry"

ADAPTER_ID = "guoyww/animatediff-motion-adapter-v1-5-2"
BASE_CANDIDATES = [
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "botp/stable-diffusion-v1-5",
    "emilianJR/epiCRealism",
]

def _vae_slicing(pipe):
    """diffusers 0.40 moved this off the pipeline onto the VAE. Support both."""
    if hasattr(pipe.vae, "enable_slicing"):
        pipe.vae.enable_slicing()
    elif hasattr(pipe, "enable_vae_slicing"):
        pipe.enable_vae_slicing()

def build_pipe(vae_slicing=True):
    adapter = MotionAdapter.from_pretrained(ADAPTER_ID, torch_dtype=torch.float16)
    last = None
    for base in BASE_CANDIDATES:
        try:
            pipe = AnimateDiffPipeline.from_pretrained(
                base, motion_adapter=adapter, torch_dtype=torch.float16)
            break
        except Exception as e:
            last = e
            print(f"  [skip] {base}: {type(e).__name__}")
    else:
        raise RuntimeError(f"No SD1.5 mirror loaded. Last error: {last}")
    pipe.scheduler = DDIMScheduler.from_pretrained(
        base, subfolder="scheduler", clip_sample=False,
        timestep_spacing="linspace", beta_schedule="linear", steps_offset=1)
    pipe.set_progress_bar_config(disable=True)
    pipe.to("cuda")
    if vae_slicing:
        _vae_slicing(pipe)
    return pipe, base

if "pipe" in globals():
    print("reusing pipe already in memory")
else:
    print("building pipeline ...")
    pipe, BASE_ID = build_pipe()
    print(f"loaded {BASE_ID}")

# One noise tensor, generated on CPU so it is identical on T4 / P100 / any GPU.
# Roadmap 2.3 requires a fixed *noise tensor*, which is stricter than a fixed seed.
shape = (1, pipe.unet.config.in_channels, FRAMES,
         H // pipe.vae_scale_factor, W // pipe.vae_scale_factor)
LATENTS = torch.randn(shape, generator=torch.Generator("cpu").manual_seed(SEED),
                      dtype=torch.float16).to("cuda")
print(f"fixed noise tensor {tuple(LATENTS.shape)}  sum={LATENTS.float().sum():.4f}")

# base, then one word changed at a time. Subjects ordered by shape-distance from
# the base, because the roadmap predicts far-shaped nouns drag the layout.
CLIPS = [
    ("BASE",         "golden retriever", "running"),
    ("noun:cat",     "cat",              "running"),
    ("noun:horse",   "horse",            "running"),
    ("noun:bird",    "bird",             "running"),
    ("verb:walking", "golden retriever", "walking"),
    ("verb:jumping", "golden retriever", "jumping"),
    ("verb:standing","golden retriever", "standing"),
]

results = {}
t_start = time.time()
for tag, subj, verb in CLIPS:
    prompt = f"a {subj} {verb} {SCENE}, high quality, detailed"
    gc.collect(); torch.cuda.empty_cache()
    t0 = time.perf_counter()
    out = pipe(prompt=prompt, negative_prompt=NEGATIVE, num_frames=FRAMES,
               height=H, width=W, num_inference_steps=STEPS, guidance_scale=CFG,
               latents=LATENTS.clone())       # clone: pipeline may edit in place
    results[tag] = {"frames": out.frames[0], "prompt": prompt,
                    "subject": subj, "verb": verb}
    print(f"{tag:<16} {time.perf_counter()-t0:6.1f} s   {prompt}")

print(f"\ntotal {(time.time()-t_start)/60:.1f} min for {len(CLIPS)} clips")


In [ ]:
# ============ E2 metrics ============
# Two distances against BASE, per clip:
#   d_c  appearance  = 1 - mean per-frame CLIP image-image cosine similarity
#   d_m  motion      = 1 - mean cosine similarity of frame-to-frame difference maps
# A noun swap should move d_c and leave d_m alone; a verb swap the reverse.

from transformers import CLIPModel, CLIPProcessor
from skimage.metrics import structural_similarity as ssim
import torch.nn.functional as F
import numpy as np, torch

_cm = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda").eval()
_cp = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

PROJ_DIM   = _cm.config.projection_dim              # 512
HIDDEN_DIM = _cm.config.vision_config.hidden_size   # 768

def _to_embedding(out):
    """transformers has moved this return type around; decide by shape, not version."""
    if torch.is_tensor(out):
        cand = out
    else:
        cand = None
        for attr in ("image_embeds", "pooler_output", "last_hidden_state"):
            v = getattr(out, attr, None)
            if torch.is_tensor(v):
                cand = v
                break
        if cand is None:
            raise TypeError(f"no tensor found on {type(out).__name__}")
    if cand.ndim == 3:                  # (B, tokens, D) -> CLS token
        cand = cand[:, 0]
    if cand.shape[-1] == HIDDEN_DIM:    # unprojected -> map into joint space
        cand = _cm.visual_projection(cand)
    return cand                          # already PROJ_DIM otherwise

def appearance_emb(frames):
    with torch.no_grad():
        inp = _cp(images=frames, return_tensors="pt").to("cuda")
        emb = _to_embedding(_cm.get_image_features(**inp))
    return F.normalize(emb.float(), dim=-1)

def motion_sig(frames, g=32):
    """Frame-to-frame difference maps: a crude 'how it moves' descriptor."""
    a = np.stack([np.asarray(f.convert("L").resize((g, g)), np.float32) / 255
                  for f in frames])
    d = np.diff(a, axis=0).reshape(len(a) - 1, -1)
    n = np.linalg.norm(d, axis=1, keepdims=True)
    return d / np.clip(n, 1e-8, None)

for tag, r in results.items():
    r["emb"] = appearance_emb(r["frames"])
    r["sig"] = motion_sig(r["frames"])
    r["gray0"] = np.asarray(r["frames"][0].convert("L"), np.float32) / 255

_e = results["BASE"]["emb"]
print(f"embedding {tuple(_e.shape)}  (proj_dim {PROJ_DIM})  "
      f"unit-norm {float(_e.norm(dim=-1).mean()):.4f}\n")

B = results["BASE"]
print(f"{'clip':<16}{'d_c appear':>12}{'d_m motion':>12}{'S index':>10}{'layout':>9}")
print("-" * 59)
rows = []
for tag, r in results.items():
    if tag == "BASE": continue
    d_c = 1 - float((r["emb"] * B["emb"]).sum(-1).mean())
    d_m = 1 - float((r["sig"] * B["sig"]).sum(-1).mean())
    S   = (d_c - d_m) / (d_c + d_m + 1e-9)     # same sign convention as S(L,t)
    lay = ssim(r["gray0"], B["gray0"], data_range=1.0)
    rows.append({"clip": tag, "d_c": round(d_c, 4), "d_m": round(d_m, 4),
                 "S": round(S, 3), "layout_ssim": round(float(lay), 3),
                 "prompt": r["prompt"]})
    print(f"{tag:<16}{d_c:>12.4f}{d_m:>12.4f}{S:>10.3f}{lay:>9.3f}")

print("\nS > 0  -> the swap moved APPEARANCE (expected for noun swaps)")
print("S < 0  -> the swap moved MOTION     (expected for verb swaps)")
print("layout_ssim near 1 -> composition stayed put under the fixed noise tensor")

noun = [r for r in rows if r["clip"].startswith("noun")]
verb = [r for r in rows if r["clip"].startswith("verb")]
mn, mv = np.mean([r["S"] for r in noun]), np.mean([r["S"] for r in verb])
print(f"\nmean S  noun swaps {mn:+.3f}   verb swaps {mv:+.3f}")
print("E2 PASSES" if mn > 0 > mv else "E2 INCONCLUSIVE - signs did not separate")


In [ ]:
# ============ E2 artifacts ============
from PIL import Image
from diffusers.utils import export_to_gif
import os, datetime

os.makedirs("/kaggle/working/E2", exist_ok=True)
for tag, r in results.items():
    export_to_gif(r["frames"], f"/kaggle/working/E2/{tag.replace(':','_')}.gif")

# contact sheet: one row per clip, 4 sampled frames -> the qualitative grid (S7)
PICK, TH = [0, 5, 10, 15], 160
sheet = Image.new("RGB", (TH * len(PICK), TH * len(results)), "white")
for ri, (tag, r) in enumerate(results.items()):
    for ci, fi in enumerate(PICK):
        sheet.paste(r["frames"][fi].resize((TH, TH)), (ci * TH, ri * TH))
sheet.save("/kaggle/working/E2/E2_contact_sheet.png")
print("contact sheet rows, top to bottom:", list(results.keys()))

json.dump({"experiment": "E2", "date": datetime.date.today().isoformat(),
           "seed": SEED, "noise": "fixed CPU-generated tensor, shared by all clips",
           "frames": FRAMES, "resolution": f"{H}x{W}", "steps": STEPS,
           "guidance_scale": CFG, "scene": SCENE, "rows": rows},
          open("/kaggle/working/E2/E2_result.json", "w"), indent=2)
print(json.dumps(rows, indent=2))
display(sheet)
